# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de Google Gemini

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

### 1.1 Instalación de dependencias

Primero instalaremos las librerías necesarias

In [1]:
!pip install google-generativeai

Defaulting to user installation because normal site-packages is not writeable
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.3 MB ? eta -:--:--
   ----------------------- ---------------- 0.8/1.3 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 2.7 MB/s  0:00:00
   ---------------------------------------- 0.0/15.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/15.6 MB 7.8 MB/s eta 0:00:02
   ------- -------------------------------- 2.9/15.6 MB 7.2 MB/s eta 0:00:02
   ----------- ---------------------------- 4.5/15.6 MB 7.3 MB/s eta 0:00:02
   -------------- ----------


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\luis3\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### 1.2 Importar librerías

In [ ]:
import google.generativeai as genai
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np
import os

# Configura tu API key de Google Gemini
# Puedes obtenerla en: https://makersuite.google.com/app/apikey
API_KEY = ""  # Reemplaza con tu API key
genai.configure(api_key=API_KEY)

print("✓ Librerías importadas correctamente")

C:\Users\luis3\AppData\Local\Temp\ipykernel_12572\3692718087.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


: 

### 1.3 Conexión básica con la API

Ejemplo simple de cómo hacer una petición a la API

In [2]:
# Listar modelos disponibles
models = genai.list_models()
print("Modelos disponibles:")
for model in models:
    print(f"  - {model.name}")

# Hacer una consulta simple
model = genai.GenerativeModel('gemini-pro')
response = model.generate_content("Hola, ¿cuál es el resultado de 2 + 2?")
print(f"\nRespuesta: {response.text}")

## 2. Retrieval (Recuperación de Información con RAG)

### 2.1 Cargo el corpus de 20 News Groups

In [8]:
# Descargar el dataset 20 News Groups
print("Descargando dataset 20 News Groups...")
newsgroups = fetch_20newsgroups(
    subset='train',
    categories=['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med'],
    remove=('headers', 'footers', 'quotes')
)

# Tomar solo los primeros 100 documentos para que sea más rápido
documents = newsgroups.data[:100]
print(f"✓ Dataset cargado: {len(documents)} documentos")
print(f"\nPrimer documento (primeras 200 caracteres):")
print(documents[0][:200] + "...")

### 2.2 Transformo a embeddings


In [3]:
# Cargar un modelo para generar embeddings
print("Cargando modelo de embeddings...")
embedding_model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

# Generar embeddings para todos los documentos
print("Generando embeddings para los documentos...")
embeddings = embedding_model.encode(documents, show_progress_bar=True)

print(f"\n✓ Embeddings generados")
print(f"  - Número de documentos: {embeddings.shape[0]}")
print(f"  - Dimensión de cada embedding: {embeddings.shape[1]}")

### 2.3 Creo una query y hago la búsqueda

In [4]:
# Crear una query
query = "¿Cuáles son los tratamientos para enfermedades del corazón?"

# Generar embedding para la query
query_embedding = embedding_model.encode([query])

print(f"Query: {query}")
print(f"Embedding de la query generado: {query_embedding.shape}")

### 2.4 Obtengo los 5 documentos más similares a mi query

In [5]:
# Calcular similaridad del coseno entre la query y todos los documentos
similarities = cosine_similarity(query_embedding, embeddings)[0]

# Obtener los índices de los 5 documentos más similares
top_5_indices = np.argsort(similarities)[-5:][::-1]

print(f"\n{'='*80}")
print(f"TOP 5 DOCUMENTOS MÁS SIMILARES A LA QUERY")
print(f"{'='*80}\n")

for i, idx in enumerate(top_5_indices, 1):
    similarity_score = similarities[idx]
    document = documents[idx]
    print(f"\n{i}. Documento #{idx} (Similaridad: {similarity_score:.4f})")
    print(f"{'-'*80}")
    # Mostrar los primeros 500 caracteres del documento
    print(document[:500])
    if len(document) > 500:
        print("...")
    print()

## 3. Uso de RAG (Retrieval-Augmented Generation)

Ahora vamos a combinar la recuperación de documentos con la generación de texto usando Gemini

In [6]:
def rag_query(query_text, top_k=3):
    """
    Función que implementa RAG:
    1. Recupera los K documentos más similares
    2. Los usa como contexto para Gemini
    3. Genera una respuesta basada en ese contexto
    """
    # Generar embedding de la query
    query_emb = embedding_model.encode([query_text])
    
    # Calcular similaridad
    sims = cosine_similarity(query_emb, embeddings)[0]
    
    # Obtener top K documentos
    top_indices = np.argsort(sims)[-top_k:][::-1]
    
    # Construir el contexto
    context = "\n---\n".join([
        f"Documento {i+1}:\n{documents[idx][:300]}"
        for i, idx in enumerate(top_indices)
    ])
    
    # Crear el prompt para Gemini
    prompt = f"""Basándote en los siguientes documentos, responde la pregunta:

DOCUMENTOS:
{context}

---

PREGUNTA: {query_text}

RESPUESTA:"""
    
    # Generar respuesta usando Gemini
    model = genai.GenerativeModel('gemini-pro')
    response = model.generate_content(prompt)
    
    return {
        'query': query_text,
        'top_documents_indices': top_indices,
        'similarities': [sims[idx] for idx in top_indices],
        'response': response.text,
        'context': context
    }

print("✓ Función RAG definida correctamente")

### 3.1 Ejemplo de uso de RAG

In [7]:
# Hacer una query usando RAG
query = "¿Qué información hay sobre enfermedades?"
result = rag_query(query, top_k=3)

print(f"QUERY: {result['query']}")
print(f"\n{'='*80}")
print(f"DOCUMENTOS RECUPERADOS:")
print(f"{'='*80}")

for i, (idx, sim) in enumerate(zip(result['top_documents_indices'], result['similarities']), 1):
    print(f"\n{i}. Documento #{idx} (Similaridad: {sim:.4f})")

print(f"\n{'='*80}")
print(f"RESPUESTA GENERADA POR GEMINI:")
print(f"{'='*80}")
print(result['response'])

## 4. Prueba con tus propias queries

Aquí puedes experimentar con diferentes preguntas

In [ ]:
# Prueba con diferentes queries
test_queries = [
    "¿Cuáles son los temas principales de los documentos?",
    "¿Qué se habla sobre religión?",
    "¿Hay información sobre gráficos o computadoras?"
]

for query in test_queries:
    print(f"\n{'#'*80}")
    print(f"QUERY: {query}")
    print(f"{'#'*80}\n")
    
    result = rag_query(query, top_k=2)
    print(f"Respuesta:\n{result['response']}")
    print()